# Data Cleaning -- Pembrolizuamb vs. Carboplatin-Based Chemotherapy
**This notebook prepares Flatiron Health CSV files for patients with advanced urothelial cancer treated with first-line pembrolizumab or carboplatin-based chemotherapy. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorUrothelial, merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/pembro_carbo_index.csv')

In [3]:
df.head(3)

,PatientID,LineName,StartDate,avelumab_maintenance
0,F5AAF96C85477,pembro,2021-07-08,0
1,F788831A66E9A,pembro,2023-02-22,0
2,F6E944C1709E6,pembro,2020-08-12,0


In [4]:
df.shape

(3712, 4)

## Clean CSV files 

In [5]:
# Initialize class 
processor = DataProcessorUrothelial()

### Process Enhanced_AdvUrothelial.csv

In [6]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvUrothelial.csv',
                                         index_date_df = df,
                                         index_date_column = 'StartDate',
                                         drop_dates = False)

2026-03-20 21:14:01,246 - INFO - Successfully read Enhanced_AdvUrothelial.csv file with shape: (13129, 13) and unique PatientIDs: 13129
2026-03-20 21:14:01,256 - INFO - Successfully filtered Enhanced_AdvUrothelial.csv file with shape: (3712, 14) and unique PatientIDs: 3712
2026-03-20 21:14:01,274 - INFO - Successfully processed Enhanced_AdvUrothelial.csv file with final shape: (3712, 16) and unique PatientIDs: 3712


In [7]:
enhanced_df['days_adv_to_treatment'] = (enhanced_df['imported_StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_adv_to_treatment'] < 30, 1, 0)

In [8]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [9]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
unknown    1808
IV         1253
II          294
III         267
I            67
0            23
Name: count, dtype: int64

In [10]:
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [11]:
enhanced_df['PrimarySite_lower'] = enhanced_df['PrimarySite'].isin(['Bladder', 'Urethra']).astype('int64')
enhanced_df = enhanced_df.drop(columns = ['PrimarySite'])

In [12]:
enhanced_df['adv_diagnosis_year'] = pd.to_numeric(enhanced_df['adv_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['adv_diagnosis_year'] < 2020, 1, 0)

In [13]:
# drop dates
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate',
                                          'AdvancedDiagnosisDate',
                                          'SurgeryDate',
                                          'imported_StartDate'])

### Process Demographics.csv 

In [14]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2025-11-26 22:59:13,937 - INFO - Successfully read Demographics.csv file with shape: (13129, 6) and unique PatientIDs: 13129
2025-11-26 22:59:13,947 - INFO - Successfully processed Demographics.csv file with final shape: (3712, 6) and unique PatientIDs: 3712


In [15]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      2712
F       999
NaN       1
Name: count, dtype: int64

In [16]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [17]:
demographics_df = demographics_df.drop(columns = 'Gender')

### Process Enhanced_AdvUrothelialBiomarkers.csv

In [18]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2025-11-26 22:59:13,977 - INFO - Successfully read Enhanced_AdvUrothelialBiomarkers.csv file with shape: (9924, 19) and unique PatientIDs: 4251
2025-11-26 22:59:13,985 - INFO - Successfully merged Enhanced_AdvUrothelialBiomarkers.csv df with index_date_df resulting in shape: (3886, 20) and unique PatientIDs: 1614
2025-11-26 22:59:14,023 - INFO - Successfully processed Enhanced_AdvUrothelialBiomarkers.csv file with final shape: (3712, 4) and unique PatientIDs: 3712


In [19]:
def map_pdl1(value):
    if pd.isna(value):  # leave missing as is
        return value
    elif value in ['0%', '< 1%']:
        return '0%'
    else:
        return '>=1%'

biomarkers_df['PDL1_binary'] = biomarkers_df['PDL1_percent_staining'].apply(map_pdl1)

In [20]:
biomarkers_df.PDL1_binary.value_counts(dropna = False)

PDL1_binary
NaN     3646
>=1%      66
Name: count, dtype: int64

In [21]:
biomarkers_df['FGFR_status'] = biomarkers_df['FGFR_status'].fillna('unknown')
biomarkers_df['PDL1_status'] = biomarkers_df['PDL1_status'].fillna('unknown')

### Process ECOG.csv

In [22]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2025-11-26 22:59:14,100 - INFO - Successfully read ECOG.csv file with shape: (184794, 4) and unique PatientIDs: 9933
2025-11-26 22:59:14,131 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (66093, 5) and unique PatientIDs: 3110
2025-11-26 22:59:14,184 - INFO - Successfully processed ECOG.csv file with final shape: (3712, 3) and unique PatientIDs: 3712


In [23]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
1      1186
NaN    1132
0       782
2       485
3       121
4         6
Name: count, dtype: int64

In [24]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [25]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [26]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2025-11-26 22:59:17,580 - INFO - Successfully read Vitals.csv file with shape: (3604484, 16) and unique PatientIDs: 13109
2025-11-26 22:59:18,958 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (1122931, 17) and unique PatientIDs: 3712
2025-11-26 22:59:19,492 - INFO - Successfully processed Vitals.csv file with final shape: (3712, 8) and unique PatientIDs: 3712


### Process Lab.csv

In [27]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2025-11-26 22:59:31,644 - INFO - Successfully read Lab.csv file with shape: (9373598, 17) and unique PatientIDs: 12700
2025-11-26 22:59:33,900 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (3090751, 18) and unique PatientIDs: 3683
2025-11-26 22:59:40,682 - INFO - Successfully processed Lab.csv file with final shape: (3712, 76) and unique PatientIDs: 3712


### Process MedicationAdministration.csv

In [28]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2025-11-26 22:59:41,942 - INFO - Successfully read MedicationAdministration.csv file with shape: (997836, 11) and unique PatientIDs: 10983
2025-11-26 22:59:42,212 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (297479, 12) and unique PatientIDs: 3656
2025-11-26 22:59:42,250 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (3712, 9) and unique PatientIDs: 3712


### Process Diagnosis.csv

In [29]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2025-11-26 22:59:42,661 - INFO - Successfully read Diagnosis.csv file with shape: (625348, 6) and unique PatientIDs: 13129
2025-11-26 22:59:42,743 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (167901, 7) and unique PatientIDs: 3712
2025-11-26 22:59:43,293 - INFO - Successfully processed Diagnosis.csv file with final shape: (3712, 40) and unique PatientIDs: 3712


In [30]:
diagnosis_df['other_gi_met'] = (
    diagnosis_df['adrenal_met'] | diagnosis_df['peritoneum_met'] | diagnosis_df['gi_met']
)

diagnosis_df['other_combined_met'] = (
    diagnosis_df['brain_met'] | diagnosis_df['other_met']
)

diagnosis_df = diagnosis_df.drop(columns = ['adrenal_met', 'peritoneum_met', 'gi_met', 'brain_met', 'other_met'])

### Process Enhanced_Mortality_V2.csv

In [31]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvUrothelialBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvUrothelial_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvUrothelial_Progression.csv',
                                           drop_dates = False)

2025-11-26 22:59:43,309 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (9040, 2) and unique PatientIDs: 9040
2025-11-26 22:59:43,317 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (3712, 3) and unique PatientIDs: 3712
2025-11-26 22:59:43,685 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2025-11-26 22:59:43,690 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (3712, 6) and unique PatientIDs: 3712. There are 0 out of 3712 patients with missing duration values


In [32]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [33]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [34]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

2025-11-26 22:59:43,783 - INFO - Successfully read Insurance.csv file with shape: (53709, 14) and unique PatientIDs: 12391
2025-11-26 22:59:43,815 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (15340, 15) and unique PatientIDs: 3519
2025-11-26 22:59:43,856 - INFO - Successfully processed Insurance.csv file with final shape: (3712, 5) and unique PatientIDs: 3712


### Process SocialDeterminantsOfHealth.csv 

In [35]:
ses_df = pd.read_csv('../data/SocialDeterminantsOfHealth.csv')

In [36]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,F5AAF96C85477,1 - Lowest SES
1,F43136CF07859,4
2,F6FAD468C5AE0,2


In [37]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
4                  2830
3                  2576
5 - Highest SES    2336
2                  2288
1 - Lowest SES     1745
NaN                1354
Name: count, dtype: int64

In [38]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 3 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(3)

In [39]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [40]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

## Merge dataframes

In [41]:
pembro_carbo_features_df = merge_dataframes(enhanced_df,
                                            demographics_df,
                                            biomarkers_df,
                                            ecog_df,
                                            vitals_df,
                                            labs_df,
                                            medications_df,
                                            diagnosis_df, 
                                            mortality_df, 
                                            insurance_df,
                                            ses_df,
                                            merge_type = 'inner')

2025-11-26 22:59:43,886 - INFO - Anticipated number of merges: 10
2025-11-26 22:59:43,887 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 162
2025-11-26 22:59:43,888 - INFO - Dataset 1 shape: (3712, 16), unique PatientIDs: 3712
2025-11-26 22:59:43,888 - INFO - Dataset 2 shape: (3712, 6), unique PatientIDs: 3712
2025-11-26 22:59:43,889 - INFO - Dataset 3 shape: (3712, 5), unique PatientIDs: 3712
2025-11-26 22:59:43,889 - INFO - Dataset 4 shape: (3712, 4), unique PatientIDs: 3712
2025-11-26 22:59:43,890 - INFO - Dataset 5 shape: (3712, 8), unique PatientIDs: 3712
2025-11-26 22:59:43,891 - INFO - Dataset 6 shape: (3712, 76), unique PatientIDs: 3712
2025-11-26 22:59:43,892 - INFO - Dataset 7 shape: (3712, 9), unique PatientIDs: 3712
2025-11-26 22:59:43,892 - INFO - Dataset 8 shape: (3712, 37), unique PatientIDs: 3712
2025-11-26 22:59:43,893 - INFO - Dataset 9 shape: (3706, 3), unique PatientIDs: 3706
2025-11-26 22:59:43,893 -

In [42]:
pembro_carbo_features_df.shape

(3706, 162)

In [43]:
pembro_carbo_features_df.head(2)

,PatientID,DiseaseGrade,SmokingStatus,GroupStage_mod,TStage_mod,NStage_mod,MStage_mod,Surgery_mod,SurgeryType_mod,days_diagnosis_to_adv,...,other_gi_met,other_combined_met,event,duration,commercial,medicaid,medicare,other_insurance,ses_mod,ses_mod_na
0,F5AAF96C85477,High grade (G2/G3/G4),History of smoking,4.0,other,unknown,unknown,0,unknown,109.0,...,0,0,0,68.0,0,0,1,0,1.0,0
1,F788831A66E9A,High grade (G2/G3/G4),History of smoking,4.0,T1,unknown,M0,1,bladder,2411.0,...,0,0,0,78.0,1,0,1,0,1.0,0


## Export dataframe

In [44]:
pembro_carbo_features_df.to_csv('../outputs/pembro_carbo_features_df.csv', index = False)

In [45]:
# Save dtypes
pembro_carbo_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/pembro_carbo_features_dtypes.csv')